# 00. YOLO26n 순수 Baseline — Local / Kaggle 공용
## 하이퍼파라미터 최적화 전 · 이미지 증강 전 기준 성능

이 Notebook은 이후 실험을 비교하기 위한 **가장 첫 기준점(Baseline)** 을 만듭니다.

권장 비교 흐름은 다음과 같습니다.

```text
A. Baseline
   YOLO26n pretrained
   + optimizer="auto"
   + 별도 HPO 없음
   + 이미지 증강 OFF
          ↓
B. Optuna 적용
   같은 데이터 / 같은 증강 OFF
   + Optuna Best hyperparameters
          ↓
C. 이미지 증강 적용
   B의 Optuna Best hyperparameters 고정
   + YOLO / OpenCV augmentation
```

이렇게 하면:

- **A → B**: 하이퍼파라미터 최적화 효과
- **B → C**: 이미지 증강 효과

를 비교하기 쉽습니다.

### `optimizer="auto"`는 무엇인가?

`auto`는 모델 종류가 아니라 **학습 optimizer 선택 방식**입니다.
Ultralytics가 학습 iteration 수 등을 보고 optimizer, 초기 learning rate, momentum을 자동으로 결정합니다.
따라서 이번 Baseline은 "Optuna 전"의 framework-default 학습 조건으로 봅니다.

### 왜 augmentation을 명시적으로 OFF 하는가?

Ultralytics Detection 기본 학습 설정에는 HSV, translation, scale, horizontal flip, mosaic 등이 이미 포함됩니다.
따라서 이미지 증강의 전/후 효과를 따로 보려면 Baseline에서는 증강을 명시적으로 꺼야 합니다.

# 1. 기대하는 데이터 구조

현재 프로젝트의 학습 데이터는 다음 구조를 사용합니다.

```text
recycling_ssg/
└─ data/
   └─ processed/
      ├─ images/
      │  ├─ train/
      │  └─ val/
      ├─ labels/
      │  ├─ train/
      │  └─ val/
      ├─ coco/
      │  ├─ instances_train.json
      │  └─ instances_val.json
      ├─ annotations/
      │  ├─ class_mapping.json
      │  └─ common_annotations.json
      ├─ manifests/
      │  ├─ image_manifest.csv
      │  ├─ sampling_summary.csv
      │  └─ sampling_shortage.csv
      └─ data.yaml
```

YOLO Detection 학습에서 직접 사용하는 핵심 입력은:

```text
images/train
images/val
labels/train
labels/val
data.yaml
```

입니다. `coco/`, `annotations/`, `manifests/`는 그대로 보존합니다.

# 2. Kaggle에 무엇을 업로드하면 되나?

가장 간단한 방법은 **프로젝트의 `data` 폴더를 Kaggle Dataset으로 업로드**하는 것입니다.

예:

```text
업로드할 Dataset
└─ data/
   └─ processed/
      ├─ images/
      ├─ labels/
      ├─ coco/
      ├─ annotations/
      ├─ manifests/
      └─ data.yaml
```

이 Notebook은 Dataset 이름을 하드코딩하지 않고 `/kaggle/input` 아래에서 `data.yaml + images/train + images/val + labels/train + labels/val` 구조를 자동 탐색합니다.

용량을 줄이고 싶다면 사실상 `data/processed`만 업로드해도 충분합니다.

# 3. 패키지 확인

필요 패키지:

- ultralytics
- torch
- pandas
- numpy
- matplotlib
- pyyaml

로컬에서는 프로젝트의 `uv` 환경을 Notebook 커널로 선택하세요.
Kaggle에서 `ultralytics`가 없다면 Internet을 켠 후 별도 설치가 필요할 수 있습니다.

예:

```python
!pip install -q -U ultralytics
```

아래 셀은 설치를 자동으로 수행하지 않고, 현재 환경을 확인합니다.

In [ ]:
from __future__ import annotations

import gc
import json
import platform
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
import ultralytics
from ultralytics import YOLO

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("Ultralytics :", ultralytics.__version__)

# 4. Local / Kaggle 환경 자동 감지

- `/kaggle/input`이 존재하면 Kaggle로 판단합니다.
- 그 외에는 Local로 판단합니다.
- Local에서는 현재 실행 폴더부터 부모 폴더를 올라가며 `data/processed`를 찾습니다.
- Kaggle에서는 `/kaggle/input` 전체에서 올바른 processed 구조를 찾습니다.

특수한 상황에서만 override 변수에 직접 경로를 넣으세요.

In [ ]:
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_WORKING_ROOT = Path("/kaggle/working")

IS_KAGGLE = KAGGLE_INPUT_ROOT.exists()
ENVIRONMENT = "KAGGLE" if IS_KAGGLE else "LOCAL"

LOCAL_PROJECT_ROOT_OVERRIDE = None
KAGGLE_PROCESSED_DIR_OVERRIDE = None

# 로컬 예시:
# LOCAL_PROJECT_ROOT_OVERRIDE = Path(r"C:\ai_challingers\recycling_ssg")

# Kaggle 예시:
# KAGGLE_PROCESSED_DIR_OVERRIDE = Path(
#     "/kaggle/input/recycling-ssg-data/data/processed"
# )


def is_valid_processed_dir(path: Path) -> bool:
    required = [
        path / "data.yaml",
        path / "images" / "train",
        path / "images" / "val",
        path / "labels" / "train",
        path / "labels" / "val",
    ]
    return all(p.exists() for p in required)


def find_local_project_root(start: Path) -> Path:
    if LOCAL_PROJECT_ROOT_OVERRIDE is not None:
        root = Path(LOCAL_PROJECT_ROOT_OVERRIDE).expanduser().resolve()
        if not is_valid_processed_dir(root / "data" / "processed"):
            raise FileNotFoundError(
                f"지정한 PROJECT_ROOT 아래 data/processed 구조가 없습니다: {root}"
            )
        return root

    start = start.expanduser().resolve()
    for candidate in [start, *start.parents]:
        if is_valid_processed_dir(candidate / "data" / "processed"):
            return candidate

    raise FileNotFoundError(
        "로컬에서 recycling_ssg/data/processed 구조를 찾지 못했습니다.\n"
        f"현재 실행 위치: {start}\n"
        "필요하면 LOCAL_PROJECT_ROOT_OVERRIDE를 직접 지정하세요."
    )


def find_kaggle_processed_dir(input_root: Path) -> Path:
    if KAGGLE_PROCESSED_DIR_OVERRIDE is not None:
        path = Path(KAGGLE_PROCESSED_DIR_OVERRIDE).resolve()
        if not is_valid_processed_dir(path):
            raise FileNotFoundError(f"잘못된 processed 경로: {path}")
        return path

    candidates = []
    for yaml_path in input_root.rglob("data.yaml"):
        parent = yaml_path.parent
        if not is_valid_processed_dir(parent):
            continue

        score = 0
        if parent.name.lower() == "processed":
            score += 10
        if parent.parent.name.lower() == "data":
            score += 3
        if (parent / "coco").exists():
            score += 1
        if (parent / "annotations").exists():
            score += 1
        if (parent / "manifests").exists():
            score += 1

        candidates.append((score, parent.resolve()))

    if not candidates:
        raise FileNotFoundError(
            "Kaggle /kaggle/input에서 YOLO processed 구조를 찾지 못했습니다.\n"
            "Dataset에 data/processed 폴더를 포함했는지 확인하세요."
        )

    candidates.sort(key=lambda item: (-item[0], str(item[1])))

    print("Kaggle processed 후보:")
    for i, (score, path) in enumerate(candidates):
        print(f"  [{i}] score={score} | {path}")

    return candidates[0][1]


if IS_KAGGLE:
    PROJECT_ROOT = None
    PROCESSED_DIR = find_kaggle_processed_dir(KAGGLE_INPUT_ROOT)
    OUTPUT_ROOT = (
        KAGGLE_WORKING_ROOT
        / "recycling_ssg"
        / "00_yolo_baseline_no_aug"
    ).resolve()
    WORKERS = 2
else:
    PROJECT_ROOT = find_local_project_root(Path.cwd())
    PROCESSED_DIR = (PROJECT_ROOT / "data" / "processed").resolve()

    if (PROJECT_ROOT / "ai").exists():
        OUTPUT_ROOT = (
            PROJECT_ROOT
            / "ai"
            / "models"
            / "yolo"
            / "00_yolo_baseline_no_aug"
        ).resolve()
    else:
        OUTPUT_ROOT = (
            PROJECT_ROOT
            / "models"
            / "yolo"
            / "00_yolo_baseline_no_aug"
        ).resolve()

    # Windows + Jupyter 안전성
    WORKERS = 0

DATA_YAML = PROCESSED_DIR / "data.yaml"
TRAIN_PROJECT = OUTPUT_ROOT / "train"
VAL_PROJECT = OUTPUT_ROOT / "validation"
REPORT_DIR = OUTPUT_ROOT / "report"
PREDICTION_DIR = OUTPUT_ROOT / "prediction_samples"

for path in [OUTPUT_ROOT, TRAIN_PROJECT, VAL_PROJECT, REPORT_DIR, PREDICTION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("ENVIRONMENT   :", ENVIRONMENT)
print("PROJECT_ROOT  :", PROJECT_ROOT)
print("PROCESSED_DIR :", PROCESSED_DIR)
print("DATA_YAML     :", DATA_YAML)
print("OUTPUT_ROOT   :", OUTPUT_ROOT)
print("WORKERS       :", WORKERS)
print("=" * 80)

# 5. GPU / Device 확인

In [ ]:
if torch.cuda.is_available():
    DEVICE = 0
    print("CUDA GPU:", torch.cuda.get_device_name(0))
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM: {total_vram:.2f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Apple MPS 사용")
else:
    DEVICE = "cpu"
    print("CPU 사용")

print("DEVICE =", DEVICE)

# 6. `data.yaml`을 현재 환경에 맞게 재작성

로컬 `data.yaml`에 Windows 절대경로가 들어 있더라도 Kaggle에서 실행 가능하도록 원본은 수정하지 않고 `runtime_data.yaml`을 별도로 만듭니다.

클래스 이름은 원래 `data.yaml`의 `names`를 그대로 사용합니다.

In [ ]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    original_yaml = yaml.safe_load(f)

if "names" not in original_yaml:
    raise KeyError("data.yaml에 names가 없습니다.")

runtime_yaml = {
    "path": str(PROCESSED_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": original_yaml["names"],
}

RUNTIME_DATA_YAML = REPORT_DIR / "runtime_data.yaml"

with open(RUNTIME_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(runtime_yaml, f, allow_unicode=True, sort_keys=False)

print(RUNTIME_DATA_YAML.read_text(encoding="utf-8"))

# 7. 학습 전 데이터 빠른 검사

여기서는 이미지/라벨 개수와 stem 1:1 매칭, YOLO label 형식을 검사합니다.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}


def list_images(directory: Path):
    return sorted(
        p for p in directory.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

train_image_dir = PROCESSED_DIR / "images" / "train"
val_image_dir = PROCESSED_DIR / "images" / "val"
train_label_dir = PROCESSED_DIR / "labels" / "train"
val_label_dir = PROCESSED_DIR / "labels" / "val"

train_images = list_images(train_image_dir)
val_images = list_images(val_image_dir)
train_labels = sorted(train_label_dir.glob("*.txt"))
val_labels = sorted(val_label_dir.glob("*.txt"))

summary_df = pd.DataFrame({
    "split": ["train", "val"],
    "images": [len(train_images), len(val_images)],
    "labels": [len(train_labels), len(val_labels)],
})
display(summary_df)


def check_pairing(images, labels, split):
    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    missing = sorted(image_stems - label_stems)
    orphan = sorted(label_stems - image_stems)
    print(f"{split}: missing_labels={len(missing)}, orphan_labels={len(orphan)}")
    return missing, orphan

train_missing, train_orphan = check_pairing(train_images, train_labels, "train")
val_missing, val_orphan = check_pairing(val_images, val_labels, "val")

if train_missing or train_orphan or val_missing or val_orphan:
    raise RuntimeError("이미지/라벨 stem 매칭 오류가 있습니다.")

In [ ]:
names_raw = original_yaml["names"]

if isinstance(names_raw, dict):
    CLASS_NAMES = {int(k): str(v) for k, v in names_raw.items()}
else:
    CLASS_NAMES = {i: str(v) for i, v in enumerate(names_raw)}

NUM_CLASSES = len(CLASS_NAMES)


def audit_labels(label_paths, split):
    errors = []
    object_count = 0

    for label_path in label_paths:
        text = label_path.read_text(encoding="utf-8").strip()
        if not text:
            errors.append({"split": split, "file": str(label_path), "error": "empty label"})
            continue

        for line_no, line in enumerate(text.splitlines(), start=1):
            parts = line.split()
            if len(parts) != 5:
                errors.append({
                    "split": split,
                    "file": str(label_path),
                    "line": line_no,
                    "error": f"expected 5 values, got {len(parts)}",
                })
                continue

            try:
                class_id_float = float(parts[0])
                class_id = int(class_id_float)
                coords = np.array([float(v) for v in parts[1:]], dtype=float)
            except ValueError as e:
                errors.append({"split": split, "file": str(label_path), "line": line_no, "error": repr(e)})
                continue

            if class_id_float != class_id or not (0 <= class_id < NUM_CLASSES):
                errors.append({"split": split, "file": str(label_path), "line": line_no, "error": f"bad class_id={parts[0]}"})

            if not np.all((coords >= 0) & (coords <= 1)):
                errors.append({"split": split, "file": str(label_path), "line": line_no, "error": f"coords outside 0..1: {coords.tolist()}"})

            if coords[2] <= 0 or coords[3] <= 0:
                errors.append({"split": split, "file": str(label_path), "line": line_no, "error": "width/height <= 0"})

            object_count += 1

    return pd.DataFrame(errors), object_count

train_errors, train_objects = audit_labels(train_labels, "train")
val_errors, val_objects = audit_labels(val_labels, "val")

print("classes      :", NUM_CLASSES)
print("train objects:", train_objects)
print("val objects  :", val_objects)
print("label errors :", len(train_errors) + len(val_errors))

if len(train_errors):
    display(train_errors.head(30))
if len(val_errors):
    display(val_errors.head(30))

if len(train_errors) or len(val_errors):
    raise RuntimeError("YOLO label 오류가 있으므로 학습을 중단합니다.")

# 8. Baseline 공통 실험 조건

이 값은 이후 Optuna 전/후 및 augmentation 전/후 비교에서도 가능하면 동일하게 유지하세요.

```text
MODEL   = yolo26n.pt
EPOCHS  = 15
IMGSZ   = 640
BATCH   = 8
SEED    = 42
```

이번 Baseline은 `optimizer="auto"`를 사용하며 사람이 `lr0`, `momentum` 등을 튜닝하지 않습니다.

In [ ]:
MODEL_NAME = "yolo26n.pt"
EPOCHS = 15
IMGSZ = 640
BATCH = 8
PATIENCE = 12
SEED = 42
OPTIMIZER = "auto"
RUN_NAME = "baseline_no_aug_seed42"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("MODEL_NAME:", MODEL_NAME)
print("EPOCHS    :", EPOCHS)
print("IMGSZ     :", IMGSZ)
print("BATCH     :", BATCH)
print("OPTIMIZER :", OPTIMIZER)
print("SEED      :", SEED)

# 9. Baseline에서는 이미지 증강 OFF

Detection에서 조절 가능한 주요 증강을 모두 0으로 만들고,
`augmentations=[]`로 Ultralytics 기본 Albumentations pipeline도 빈 목록으로 대체합니다.

이 설정이 중요한 이유는 나중에 이미지 증강 실험과 공정하게 비교하기 위해서입니다.

In [ ]:
NO_AUGMENTATION = {
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.0,
    "degrees": 0.0,
    "translate": 0.0,
    "scale": 0.0,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.0,
    "bgr": 0.0,
    "mosaic": 0.0,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "close_mosaic": 0,
    "erasing": 0.0,
    "auto_augment": None,
    "augmentations": [],
}

display(pd.DataFrame({
    "augmentation_arg": list(NO_AUGMENTATION.keys()),
    "value": [repr(v) for v in NO_AUGMENTATION.values()],
}))

# 10. YOLO26n Baseline 학습

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model = YOLO(MODEL_NAME)
started = time.perf_counter()

train_results = model.train(
    data=str(RUNTIME_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=DEVICE,
    workers=WORKERS,
    optimizer=OPTIMIZER,
    seed=SEED,
    deterministic=True,
    project=str(TRAIN_PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    save=True,
    plots=True,
    verbose=True,
    **NO_AUGMENTATION,
)

train_minutes = (time.perf_counter() - started) / 60.0
TRAIN_SAVE_DIR = Path(model.trainer.save_dir)
BEST_PT = TRAIN_SAVE_DIR / "weights" / "best.pt"
LAST_PT = TRAIN_SAVE_DIR / "weights" / "last.pt"
ACTUAL_OPTIMIZER_CLASS = type(model.trainer.optimizer).__name__

print("train_minutes          :", train_minutes)
print("TRAIN_SAVE_DIR         :", TRAIN_SAVE_DIR)
print("BEST_PT                :", BEST_PT)
print("ACTUAL_OPTIMIZER_CLASS :", ACTUAL_OPTIMIZER_CLASS)

if not BEST_PT.exists():
    raise FileNotFoundError(BEST_PT)

# 11. `optimizer="auto"`가 실제 무엇을 선택했는지 기록

`optimizer="auto"`는 `auto`라는 optimizer로 학습하는 것이 아닙니다.
Ultralytics가 실제 optimizer를 선택합니다.

위 셀의 `ACTUAL_OPTIMIZER_CLASS`와 학습 로그를 확인하세요.
또 `args.yaml`을 그대로 보존합니다.

In [ ]:
ARGS_YAML = TRAIN_SAVE_DIR / "args.yaml"

if ARGS_YAML.exists():
    print(ARGS_YAML.read_text(encoding="utf-8")[:10000])
else:
    print("args.yaml not found")

# 12. Best checkpoint로 Validation

마지막 epoch가 아니라 학습 중 Validation 성능이 가장 좋았던 `best.pt`를 다시 로드합니다.
`plots=True`로 PR/F1/Precision/Recall curve와 confusion matrix도 저장합니다.

In [ ]:
best_model = YOLO(str(BEST_PT))

val_metrics = best_model.val(
    data=str(RUNTIME_DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    plots=True,
    project=str(VAL_PROJECT),
    name="baseline_no_aug_validation",
    exist_ok=True,
    verbose=True,
)

VAL_SAVE_DIR = Path(val_metrics.save_dir)
print("VAL_SAVE_DIR:", VAL_SAVE_DIR)

# 13. 핵심 성능지표 추출

향후 Optuna/augmentation 실험과 다음 지표를 동일한 방식으로 비교합니다.

- Precision
- Recall
- F1
- mAP50
- mAP75
- mAP50-95
- inference ms/image

In [ ]:
def extract_metrics(metrics) -> dict:
    box = metrics.box
    precision = float(getattr(box, "mp", np.nan))
    recall = float(getattr(box, "mr", np.nan))

    if np.isfinite(precision) and np.isfinite(recall) and precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    speed = getattr(metrics, "speed", {}) or {}

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mAP50": float(box.map50),
        "mAP75": float(box.map75),
        "mAP50_95": float(box.map),
        "inference_ms_per_image": speed.get("inference", np.nan),
    }

baseline_metrics = extract_metrics(val_metrics)
baseline_metrics

# 14. Baseline 요약 CSV 저장

In [ ]:
actual_epochs = int(getattr(model.trainer, "epoch", -1)) + 1

baseline_summary = {
    "experiment": "baseline_no_aug",
    "model": MODEL_NAME,
    "optimizer_requested": OPTIMIZER,
    "optimizer_actual_class": ACTUAL_OPTIMIZER_CLASS,
    "hyperparameter_optimization": False,
    "image_augmentation": False,
    "epochs_requested": EPOCHS,
    "epochs_actual": actual_epochs,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "train_images": len(train_images),
    "val_images": len(val_images),
    "train_objects": train_objects,
    "val_objects": val_objects,
    "train_minutes": train_minutes,
    "best_pt": str(BEST_PT),
    **baseline_metrics,
}

baseline_summary_df = pd.DataFrame([baseline_summary])
BASELINE_SUMMARY_CSV = REPORT_DIR / "baseline_summary.csv"
baseline_summary_df.to_csv(BASELINE_SUMMARY_CSV, index=False, encoding="utf-8-sig")

display(baseline_summary_df.T)
print("Saved:", BASELINE_SUMMARY_CSV)

# 15. 핵심 성능지표 시각화

In [ ]:
metric_names = ["precision", "recall", "f1", "mAP50", "mAP50_95"]
metric_values = [baseline_metrics[name] for name in metric_names]

plt.figure(figsize=(9, 5))
plt.bar(metric_names, metric_values)
plt.ylim(0, max(1.0, max(metric_values) * 1.15))
plt.ylabel("Score")
plt.title("YOLO26n Baseline - No Augmentation")
plt.tight_layout()

metric_plot_path = REPORT_DIR / "baseline_metrics.png"
plt.savefig(metric_plot_path, dpi=170, bbox_inches="tight")
plt.show()
print("Saved:", metric_plot_path)

# 16. Epoch별 loss / metric 확인

In [ ]:
RESULTS_CSV = TRAIN_SAVE_DIR / "results.csv"

if RESULTS_CSV.exists():
    history_df = pd.read_csv(RESULTS_CSV)
    history_df.columns = [c.strip() for c in history_df.columns]
    display(history_df.tail())
else:
    history_df = pd.DataFrame()
    print("results.csv not found")

In [ ]:
if len(history_df):
    loss_columns = [c for c in history_df.columns if "loss" in c.lower()]

    if loss_columns:
        plt.figure(figsize=(11, 6))
        for column in loss_columns:
            plt.plot(history_df.index + 1, history_df[column], label=column)
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Baseline Training / Validation Loss")
        plt.legend()
        plt.tight_layout()
        path = REPORT_DIR / "loss_curves.png"
        plt.savefig(path, dpi=170, bbox_inches="tight")
        plt.show()
        print("Saved:", path)

In [ ]:
if len(history_df):
    tokens = ["precision", "recall", "map50", "map50-95"]
    metric_columns = []

    for column in history_df.columns:
        normalized = column.lower().replace(" ", "")
        if any(token in normalized for token in tokens):
            metric_columns.append(column)

    if metric_columns:
        plt.figure(figsize=(11, 6))
        for column in metric_columns:
            plt.plot(history_df.index + 1, history_df[column], label=column)
        plt.xlabel("Epoch")
        plt.ylabel("Metric")
        plt.title("Baseline Validation Metrics by Epoch")
        plt.legend()
        plt.tight_layout()
        path = REPORT_DIR / "epoch_metrics.png"
        plt.savefig(path, dpi=170, bbox_inches="tight")
        plt.show()
        print("Saved:", path)

# 17. 클래스별 mAP50-95 저장

In [ ]:
per_class_maps = np.asarray(val_metrics.box.maps, dtype=float)
per_class_rows = []

for class_id, class_name in CLASS_NAMES.items():
    class_map = float(per_class_maps[class_id]) if class_id < len(per_class_maps) else np.nan
    per_class_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "mAP50_95": class_map,
    })

per_class_df = pd.DataFrame(per_class_rows)
PER_CLASS_CSV = REPORT_DIR / "baseline_per_class_map.csv"
per_class_df.to_csv(PER_CLASS_CSV, index=False, encoding="utf-8-sig")

display(per_class_df.sort_values("mAP50_95", ascending=False))
print("Saved:", PER_CLASS_CSV)

In [ ]:
plot_df = per_class_df.dropna(subset=["mAP50_95"]).sort_values("mAP50_95")

if len(plot_df):
    plt.figure(figsize=(10, max(8, len(plot_df) * 0.22)))
    plt.barh(plot_df["class_name"], plot_df["mAP50_95"])
    plt.xlabel("Validation mAP50-95")
    plt.title("Baseline Per-Class mAP50-95")
    plt.tight_layout()
    path = REPORT_DIR / "baseline_per_class_map.png"
    plt.savefig(path, dpi=170, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

# 18. 실제 Validation 예측 샘플

숫자만 보지 않고 최대 12개 Validation 이미지에 대한 Detection 결과도 저장합니다.

In [ ]:
sample_images = [str(path) for path in val_images[:12]]

if sample_images:
    best_model.predict(
        source=sample_images,
        imgsz=IMGSZ,
        conf=0.25,
        device=DEVICE,
        save=True,
        project=str(PREDICTION_DIR),
        name="baseline",
        exist_ok=True,
        verbose=False,
    )

    print("Prediction samples:", PREDICTION_DIR / "baseline")

# 19. 실험 설정 JSON 저장

In [ ]:
BASELINE_CONFIG_JSON = REPORT_DIR / "baseline_config.json"

baseline_config = {
    "environment": ENVIRONMENT,
    "ultralytics_version": ultralytics.__version__,
    "torch_version": torch.__version__,
    "model": MODEL_NAME,
    "pretrained": True,
    "optimizer_requested": OPTIMIZER,
    "optimizer_actual_class": ACTUAL_OPTIMIZER_CLASS,
    "hyperparameter_optimization": False,
    "augmentation": False,
    "augmentation_args": NO_AUGMENTATION,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "patience": PATIENCE,
    "seed": SEED,
    "processed_dir": str(PROCESSED_DIR),
    "runtime_data_yaml": str(RUNTIME_DATA_YAML),
    "best_pt": str(BEST_PT),
    "metrics": baseline_metrics,
}

with open(BASELINE_CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(baseline_config, f, ensure_ascii=False, indent=2, default=str)

print(BASELINE_CONFIG_JSON.read_text(encoding="utf-8"))

# 20. 최종 산출물 구조

로컬에서는 기본적으로:

```text
recycling_ssg/
└─ ai/models/yolo/00_yolo_baseline_no_aug/
```

Kaggle에서는:

```text
/kaggle/working/recycling_ssg/00_yolo_baseline_no_aug/
```

아래에 다음 결과가 생성됩니다.

```text
00_yolo_baseline_no_aug/
├─ train/
│  └─ baseline_no_aug_seed42/
│     ├─ weights/
│     │  ├─ best.pt
│     │  └─ last.pt
│     ├─ results.csv
│     ├─ results.png
│     └─ args.yaml
├─ validation/
│  └─ baseline_no_aug_validation/
│     ├─ PR_curve.png
│     ├─ F1_curve.png
│     ├─ confusion_matrix.png
│     └─ ...
├─ prediction_samples/
└─ report/
   ├─ baseline_summary.csv
   ├─ baseline_config.json
   ├─ baseline_metrics.png
   ├─ baseline_per_class_map.csv
   ├─ baseline_per_class_map.png
   ├─ loss_curves.png
   └─ epoch_metrics.png
```

나중에 비교할 때 가장 먼저 볼 파일은 `report/baseline_summary.csv`입니다.

# 21. 다음 실험에서 지켜야 할 비교 원칙

## ① 하이퍼파라미터 최적화 전/후

```text
Baseline
optimizer=auto
augmentation OFF

vs

Optuna Best
optimized optimizer/lr/weight_decay/...
augmentation OFF
```

즉 **Optuna Notebook에서도 증강을 OFF 해야 순수 HPO 효과를 비교**할 수 있습니다.

## ② 이미지 증강 전/후

```text
Optuna Best hyperparameters
augmentation OFF

vs

똑같은 Optuna Best hyperparameters
augmentation ON
```

즉 2번 이미지 증강 Notebook에서는 Optuna 결과를 고정한 뒤 augmentation만 바꾸는 것이 가장 공정합니다.

가능하면 다음도 동일하게 유지하세요.

```text
train/val split
model=yolo26n
imgsz=640
batch=8
epochs=15
seed
evaluation code
Validation dataset
```